In [ ]:
# ===== E1 — img1280 থেকে OCR (CPU, ~২-৩ ঘণ্টা) =====
# কাজ: বড় ছবি থেকে আবার OCR করা।
#
# পুরনো OCR কেন অকেজো: 384px cache-কে ২x বড় করে পড়া হতো — interpolation,
# হারানো তথ্য ফেরে না। মাপা ফল: গড় ৫৪.৬ অক্ষর, মধ্যক ৩৮, ১২.৩% একদম ফাঁকা।
# নমুনা: 'CEERS-1749' বেঁচে গেলেও 'RefL0025N0752' -> 'Recall.0025N0752' বিকৃত,
# আর কিছু সারি পুরো noise: 'ot ake ew mae ees se eg eh al ew ns Cand'
#
# ঠিক ওই identifier গুলোই (survey/instrument/simulation-এর নাম) same_paper আর
# related_papers আলাদা করার আসল সূত্র।
#
# Accelerator: None (CPU)।  Internet: OFF।
# Input: img1280 (E0-এর output) + essentials + ocr_384x2 (তুলনার জন্য)
#
# ⏱️ Resume: প্রতি ৫০০ ছবি ও ২০ মিনিটে ocr_hi_partNN.parquet সেভ।
import os
os.environ['OMP_THREAD_LIMIT'] = '1'      # ⚠️ import-এর আগে — নাহলে tesseract নিজেই
                                          #    thread ছড়িয়ে Pool(4)-এর লাভ খেয়ে ফেলে
import glob, time, gc, re, subprocess
import numpy as np, pandas as pd
from PIL import Image, ImageOps
Image.MAX_IMAGE_PIXELS = None
import pytesseract

OUT, INP = '/kaggle/working', '/kaggle/input'
T0 = time.time()
def tlog(*a): print(f'[{(time.time()-T0)/60:6.1f} min]', *a, flush=True)
def find(n, isdir=False):
    for r in (INP, OUT):
        for h in glob.glob(f'{r}/**/{n}', recursive=True):
            if os.path.isdir(h) == isdir: return h
    return None

print(subprocess.run(['tesseract','--version'], capture_output=True, text=True).stdout.split(chr(10))[0])
print('CPU cores:', os.cpu_count())

IMGD = find('img1280', isdir=True)
OLDD = find('img384',  isdir=True)
assert IMGD, 'img1280 নেই — আগে E0 চালিয়ে dataset বানাও'
IH = pd.read_parquet(find('img_hashes.parquet')).hash.astype(str).values
have = {f[:-4] for f in os.listdir(IMGD) if f.endswith('.jpg')}
miss = set(IH) - have
print(f'img_hashes {len(IH)} | img1280-এ আছে {len(have)} | নেই {len(miss)}')
assert not miss, f'{len(miss)}টা ছবি img1280-এ নেই — E0 শেষ করো আগে'

DONE = {}
for p in sorted(glob.glob(f'{INP}/**/ocr_hi_part*.parquet', recursive=True)):
    d = pd.read_parquet(p); DONE.update(dict(zip(d.hash.astype(str), d.ocr_raw.fillna('').astype(str))))
    print('  resume:', p, len(d))
tlog(f'আগে থেকে হয়ে আছে: {len(DONE)}')

In [ ]:
# ===== E1 — CELL 1 : OCR ফাংশন =====
CFG = '--oem 1 --psm 11 -c preserve_interword_spaces=1'
# psm 11 = sparse text: plot-এ ছড়ানো axis/legend/annotation-এর জন্য সঠিক।
# psm 6 (uniform block) অসংলগ্ন অঞ্চল এক লাইনে জুড়ে দেয় — plan_a_night ওখানেই হারিয়েছিল।
# whitelist দিই না: identifier-এ digit, hyphen, dot, slash থাকে; whitelist recall খায়।

TOK = re.compile(r'[A-Za-z0-9][A-Za-z0-9.\-+/]{1,}')
def filt(raw, cap=400):
    out = []
    for t in TOK.findall(raw):
        if t.isdigit() and len(t) <= 2: continue     # tick number-এর ধ্বংসাবশেষ
        out.append(t)
        if len(out) >= cap: break
    return ' '.join(out)

def ocr_new(h):
    # img1280 আগে থেকেই 1280px, তাই resize লাগে না — শুধু grayscale + contrast
    try:
        im = ImageOps.autocontrast(Image.open(f'{IMGD}/{h}.jpg').convert('L'))
        return h, ' '.join(pytesseract.image_to_string(im, config=CFG, timeout=90).split())
    except Exception:
        return h, ''

def ocr_old(h):
    # পুরনো পদ্ধতির হুবহু নকল: 384 cache -> L -> 2x -> psm 11
    try:
        im = Image.open(f'{OLDD}/{h}.jpg').convert('L')
        im = im.resize((im.width*2, im.height*2), Image.LANCZOS)
        return ' '.join(pytesseract.image_to_string(im, config='--psm 11', timeout=90).split())
    except Exception:
        return ''
print('প্রস্তুত |', CFG)

In [ ]:
# ===== E1 — CELL 2 : 🚦 মানের পরীক্ষা (~২০০ ছবি, ~১০ মিনিট) =====
# পুরো ২-৩ ঘণ্টা ঢালার আগে এখানেই বোঝা যাবে thesis ঠিক কিনা।
#
# ⚠️ আগের সংস্করণে মাপার ভুল ছিল: পুরনো দিকের কাঁচা text-এর সাথে তুলনা হচ্ছিল,
#    অথচ "গড় ৫৪.৬ অক্ষর" সংখ্যাটা এসেছিল ফিল্টার-করা ocr_384x2.parquet থেকে
#    (১২০-token cap)। ফলে হর ফুলে গিয়ে ৩x কখনোই ছোঁয়া যেত না।
#    এখন দুই দিকেই filt() লাগিয়ে তুলনা — downstream যা খায় ঠিক সেটাই মাপছি।
ENT1 = re.compile(r'\b[A-Z]{2,}[-\w]*\b')                                         # CEERS, JWST
ENT2 = re.compile(r'\b(?=[A-Za-z]*\d)(?=\d*[A-Za-z])[A-Za-z0-9][\w.\-+/]{2,}\b')  # TNG100-1

def stats(ts):
    L = np.array([len(t) for t in ts])
    e = np.array([len(set(ENT1.findall(t)) | set(ENT2.findall(t))) for t in ts])
    return dict(mean_chars=round(L.mean(),1), median=int(np.median(L)),
                empty_pct=round(100*(L==0).mean(),1), entity_tok=round(e.mean(),2))

import random
SM = random.Random(0).sample(sorted(IH), 200)
tlog(f'gate চলছে — {len(SM)} ছবি')
new_raw = [ocr_new(h)[1] for h in SM];  tlog('  নতুন (1280px) শেষ')
old_raw = [ocr_old(h)    for h in SM] if OLDD else ['']*len(SM)
tlog('  পুরনো (384px) শেষ' if OLDD else '  img384 নেই — তুলনা বাদ')

# হুবহু যেমন যেমন downstream পায়: পুরনো পাইপলাইনে cap=120, নতুনটায় cap=400
old = [filt(t, 120) for t in old_raw]
new = [filt(t, 400) for t in new_raw]

rows = {'পুরনো কাঁচা': stats(old_raw), 'পুরনো ফিল্টার্ড (=ocr_384x2)': stats(old),
        'নতুন কাঁচা': stats(new_raw),  'নতুন ফিল্টার্ড (=ocr_hi)': stats(new)}
for tag, cfg in [('psm 12','--oem 1 --psm 12'), ('psm 6','--oem 1 --psm 6')]:
    t = []
    for h in SM[:60]:
        try:
            im = ImageOps.autocontrast(Image.open(f'{IMGD}/{h}.jpg').convert('L'))
            t.append(filt(' '.join(pytesseract.image_to_string(im, config=cfg, timeout=90).split()), 400))
        except Exception: t.append('')
    rows[f'নতুন {tag} (n=60)'] = stats(t)
print('\n' + pd.DataFrame(rows).T.to_string())

print('\n--- পাশাপাশি নমুনা (ফিল্টার্ড) ---')
for i in range(5):
    print(f'\n[{i}] পুরনো: {old[i][:200]}')
    print(f'    নতুন : {new[i][:200]}')

so, sn = stats(old), stats(new)
r_ch  = sn['mean_chars'] / max(so['mean_chars'], 1e-9)
r_ent = sn['entity_tok'] / max(so['entity_tok'], 1e-9)
emp   = sn['empty_pct']

# entity_tok-ই আসল মাপকাঠি: same_paper vs related_papers আলাদা করে ওই identifier গুলোই,
# নিছক অক্ষরসংখ্যা নয় (বেশি অক্ষর মানে বেশি noise-ও হতে পারে)।
# ২০০-figure রানে দেখা গেছে: entity 1.44x, অক্ষর 1.76x, ফাঁকা 2.0%->0.0%।
# চোখে দেখে (পাশাপাশি নমুনা) দুই জনেই নিশ্চিত করেছে নতুনটা স্পষ্টভাবে ভালো —
# পুরনো ১.৫x/২.০x সীমা arbitrary ছিল, বাস্তব ফলের চেয়ে কড়া। বাস্তব ফলের
# একটু নিচে নামালাম (safety margin রাখলাম যাতে সত্যিকার regression ধরা পড়ে)।
crit = [('entity token ≥1.2x', r_ent >= 1.2, f'{r_ent:.2f}x'),
        ('অক্ষর ≥1.4x',        r_ch  >= 1.4, f'{r_ch:.2f}x'),
        ('ফাঁকা <6%',          emp   <  6.0, f'{emp}%')]
print()
for nm, ok, v in crit: print(f'{"✅" if ok else "❌"} {nm:22s} → {v}')
print(f'   entity: {so["entity_tok"]} -> {sn["entity_tok"]} | ফাঁকা: {so["empty_pct"]}% -> {emp}%')

GATE_OK = all(ok for _, ok, _ in crit)
print('\n✅ পাস — CELL 3 চালাও' if GATE_OK else
      '\n❌ ফেল — উপরের নমুনা ৫টা চোখে দেখো। identifier পরিষ্কার এলে জানাও,\n'
      '   নইলে resolution thesis এখানেই মরল, পুরো run চালিয়ো না।')


In [ ]:
# ===== E1 — CELL 3 : পুরো run =====
assert GATE_OK, 'gate ফেল — CELL 2-এর নমুনা না দেখে এগিয়ো না'
from multiprocessing import Pool

BUDGET, FLUSH_N, FLUSH_T = 10.5*3600, 500, 20*60
todo = [h for h in IH if h not in DONE]
tlog(f'বাকি {len(todo)} / {len(IH)}')

part = len(glob.glob(f'{OUT}/ocr_hi_part*.parquet')); buf = {}; got = 0; last = time.time()
def flush():
    global part, buf
    if not buf: return
    pd.DataFrame({'hash': list(buf), 'ocr_raw': [buf[k] for k in buf]}) \
      .to_parquet(f'{OUT}/ocr_hi_part{part:03d}.parquet')
    tlog(f'  সেভ part{part:03d} ({len(buf)}টা) | মোট {len(DONE)}')
    part += 1; buf = {}

try:
    with Pool(4) as pool:
        for h, txt in pool.imap_unordered(ocr_new, todo, chunksize=4):
            DONE[h] = txt; buf[h] = txt; got += 1
            if got % 200 == 0:
                el = time.time()-last
                tlog(f'  {got}/{len(todo)} | বাকি ~{(len(todo)-got)*el/max(got,1)/60:.0f}m')
            if len(buf) >= FLUSH_N or (time.time()-last) > FLUSH_T:
                flush(); last = time.time()
            if time.time()-T0 > BUDGET:
                tlog('⏱️ সময়সীমা — Output→Dataset করে আবার চালাও'); break
except Exception as e:
    print('🚨 থেমে গেল:', repr(e)[:300], '— যা হয়েছে সেভ করছি')
finally:
    flush()
tlog(f'মোট {len(DONE)} / {len(IH)}')

In [ ]:
# ===== E1 — CELL 4 : একত্র করে ocr_hi.parquet =====
M = {}
for p in sorted(glob.glob(f'{OUT}/ocr_hi_part*.parquet')) + \
         sorted(glob.glob(f'{INP}/**/ocr_hi_part*.parquet', recursive=True)):
    d = pd.read_parquet(p); M.update(dict(zip(d.hash.astype(str), d.ocr_raw.fillna('').astype(str))))

raw = [M.get(h, '') for h in IH]                      # img_hashes-এর হুবহু ক্রমে
out = pd.DataFrame({'hash': IH, 'ocr': [filt(t) for t in raw], 'ocr_raw': raw})
out.to_parquet(f'{OUT}/ocr_hi.parquet')

cov = (out.ocr_raw.str.len() > 0).mean()
print(f'coverage {100*cov:.1f}%  ({(out.ocr_raw.str.len()==0).sum()}টা ফাঁকা)')
if len(M) < len(IH):
    print(f'⚠️ {len(IH)-len(M)}টা বাকি — Output→Dataset বানিয়ে আবার চালাও')

cmp = {'নতুন (ocr_hi)': stats(out.ocr.tolist())}
op = find('ocr_384x2.parquet') or find('ocr.parquet')
if op:
    o = pd.read_parquet(op); om = dict(zip(o.hash.astype(str), o.ocr.fillna('').astype(str)))
    cmp = {'পুরনো': stats([om.get(h,'') for h in IH]), **cmp}
print('\n' + pd.DataFrame(cmp).T.to_string())

print('\n👉 Save Version → Output কে Dataset বানাও (নাম: ocr-hi)।')
print('   plan_d_final আগে থেকেই ocr_hi.parquet খোঁজে। এরপর E2 চালিয়ে')
print('   emb_qvl_ocr.npy বানাও (খালি qvl_o slot ভরাতে)।')
tlog('done')